<a href="https://colab.research.google.com/github/Erikiss/Interface-Neuron-Atlas-by-GoodFire-AI/blob/main/notebooks/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# Neuron Atlas Interface in Google Colab

Suchen, lesen, herunterladen und Geometrie analysieren im
[Qwen3.6 Neuron Atlas](https://www.goodfire.com/research/public_neuron_atlas) von Goodfire – ohne lokale
Installation. Die Zellen der Reihe nach ausführen (▶ oder Strg+Enter). Eine normale CPU-Laufzeit genügt.

> **Inoffiziell.** Die Daten gehören Goodfire AI. Ihre Nutzungsbedingungen erlauben persönliche,
> nicht-kommerzielle Nutzung; automatisiertes Kopieren und Weiterverbreiten nur mit Zustimmung. Das
> Werkzeug lädt deshalb nur, was du öffnest oder ausdrücklich herunterlädst – gedrosselt (4 Anfragen/s),
> jede Datei nur einmal. Mehr im [README](https://github.com/Erikiss/Interface-Neuron-Atlas-by-GoodFire-AI/blob/main/README.md#rechtliches-und-rücksicht).

## 1. Installieren

Etwa eine Minute; aus dem GitHub-Repository. Auf der aktuellen Colab-Laufzeit kommen nur das Paket und
der MCP-Server dazu, alles andere ist schon da – ein Neustart ist normalerweise nicht nötig. Fragt Colab
doch danach („Restart session“), bestätigen und ab Schritt 2 weitermachen.

**Privates Repository?** Dann einen GitHub-Token mit Lesezugriff auf das Repository (fein granuliert,
„Contents: Read-only“) als Colab-Secret `GITHUB_TOKEN` anlegen (Schlüssel-Symbol links, Zugriff für
dieses Notebook erlauben). Die Zelle reicht ihn nur als HTTP-Header an git weiter; er erscheint in keiner
Ausgabe. Bei einem öffentlichen Repository ist nichts nötig.

Später eine neuere Fassung holen: in der `%pip`-Zeile `--force-reinstall --no-deps` ergänzen, dann
*Laufzeit → Sitzung neu starten*.

In [ ]:
import base64, os

try:  # optional: a token for a private repository, from Colab's secrets
    from google.colab import userdata

    _token = userdata.get("GITHUB_TOKEN")
except Exception:
    _token = None
if _token:  # as an HTTP header for git (as GitHub Actions does it): never in a URL or in pip's output
    _basic = base64.b64encode(f"x-access-token:{_token}".encode()).decode()
    os.environ.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                       "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {_basic}"})

%pip install "neuron-atlas-interface[all] @ git+https://github.com/Erikiss/Interface-Neuron-Atlas-by-GoodFire-AI.git@main"

for _key in ("GIT_CONFIG_COUNT", "GIT_CONFIG_KEY_0", "GIT_CONFIG_VALUE_0"):
    os.environ.pop(_key, None)
del _token

In [ ]:
import importlib.metadata as md, json, sys

dist = md.distribution("neuron-atlas-interface")
commit = (json.loads(dist.read_text("direct_url.json") or "{}").get("vcs_info") or {}).get("commit_id", "")[:12]
print(f"Python {sys.version.split()[0]} · neuron-atlas-interface {dist.version} · Stand {commit or 'lokal'}")

## 2. (optional) Einen gesicherten Spiegel aus Google Drive zurückholen

Eine Colab-Sitzung ist vorübergehend: Was heruntergeladen wurde, ist beim nächsten Mal weg. Mit
`USE_DRIVE = True` holt diese Zelle einen früher gesicherten Spiegel zurück (Schritt 9 sichert ihn) –
als **eine** Archivdatei, denn tausende kleine Dateien und SQLite-Datenbanken direkt auf Drive sind
langsam und unzuverlässig. Gearbeitet wird immer auf der schnellen Platte der Sitzung.

In [ ]:
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/neuron-atlas"

import glob
from neuron_atlas import notebook

if USE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        for archive in sorted(glob.glob(f"{DRIVE_DIR}/neuron-atlas-*.tar")):
            print(archive, "->", notebook.restore(archive), "Dateien zurückgeholt")
    except Exception as exc:  # Drive kann gelegentlich nicht eingebunden werden; dann ohne weiter
        USE_DRIVE = False
        print("Google Drive nicht verfügbar:", exc)

## 3. Den Atlas öffnen

Beim ersten Mal werden sieben Index-Dateien geladen (zusammen etwa 3 MB). Danach ist der ganze Atlas
durchsuchbar: alle veröffentlichten Pole, Modi und Kennzahlen.

In [ ]:
atlas = notebook.open_atlas()
if USE_DRIVE:
    atlas.index_mirrored()  # die zurückgeholten Texte wieder durchsuchbar machen
atlas.info()["atlas"]

## 4. Suchen und lesen

Dieselbe Syntax wie in der Suchzeile der Oberfläche: Wörter, `"Phrasen"`, `-ausschließen`, `präfix*`,
`OR` / `NOT` (in Großbuchstaben), Filter wie `layer:24..36 cat:mono,fn pole:neg sim:>0.8`. Die
Beschreibungen im Atlas sind englisch.

In [ ]:
import pandas as pd

res = atlas.search({"q": "braille OR tactile", "limit": 20})
print(f"{res.total} Treffer")
pd.DataFrame([{"Pol": h.pid, "Kategorie": h.category_words, "Beschreibung": h.description} for h in res.hits])

In [ ]:
# Ein Pol als Dokument (dieselbe Ansicht wie die Druckseite); fehlende Dateien werden einmal geholt
notebook.show_pole("L30_E47_N449__pos", include=("modes", "examples", "overview"), max_examples=1)

In [ ]:
# Ähnliche Bedeutungen - zu einem Text oder zu einem Pol
sim = atlas.similar(text="random hex hashes and base64", top=10)
pd.DataFrame(
    [{"Pol": i.pid, "Modus": i.mode_index, "Ähnlichkeit": round(i.score, 3), "Text": i.label or i.description} for i in sim.items]
)

## 5. Eine semantische Region herunterladen

Erst ansehen, was geladen würde, dann laden (gedrosselt; ■ bricht ab, der nächste Lauf setzt fort).
Danach sind auch die langen Texte der Region durchsuchbar (Erklärungen, Subcircuits, Reports), und die
Geometrie-Werkzeuge haben mehr Daten. Für eine größere Region `top` erhöhen.

In [ ]:
from neuron_atlas.models import MirrorRequest, Selection

request = MirrorRequest(selection=Selection(similar_to_text="braille and dot patterns", top=10))
atlas.plan_mirror(request).estimate()

In [ ]:
plan, report = atlas.mirror(request)
report.as_dict()

## 6. Geometrie: Karten und Tabellen

Die Karte ordnet Modi nach Bedeutung an (UMAP), Farbe = Cluster; Klick auf einen Punkt zeigt den Pol.
Welche Strukturen in den Daten stecken – und welche nicht –, steht in [docs/geometry.md](https://github.com/Erikiss/Interface-Neuron-Atlas-by-GoodFire-AI/blob/main/docs/geometry.md).

In [ ]:
from neuron_atlas.geometry.maps import semantic_map
from neuron_atlas.models import SearchQuery

pids = [h.pid for h in atlas.iter_search(SearchQuery(q="hash OR base64 OR hex"))]
data = semantic_map(atlas, pids, unit="mode", method="umap", clusters=8)
print(len(data["points"]), "Punkte;", "; ".join(data["notes"]))
notebook.show_map(data, title="Hashes, Base64, Hex")

In [ ]:
# Alles Gespiegelte als Tabellen: Pole, Modi, Feuerungen, Modus×Modus-Ähnlichkeiten, Circuit-Kanten, …
from neuron_atlas.geometry.tables import write_tables

written = write_tables(atlas, "/content/tables", fmt="parquet")
print(written)
pd.read_parquet("/content/tables/modes.parquet").head()

## 7. Kommandozeile im Notebook

Alle Befehle aus dem README gehen auch hier, mit `!` davor.

In [ ]:
!neuron-atlas search "chess opening" -n 5

## 8. (optional) Die Oberfläche im Notebook

Weboberfläche, REST-API (`/docs`) und MCP-Endpunkt laufen im Hintergrund dieser Sitzung; die Seite
erscheint unter der Zelle, ein Link öffnet sie in einem eigenen Tab. Sie teilt sich den Atlas mit den
Python-Zellen oben.

- Erreichbar nur für dich, in diesem Browser und solange dieses Notebook geöffnet ist. Nach einem
  Neuverbinden die Zelle erneut ausführen – die Adresse ändert sich mit jeder Laufzeit.
- Einstellungen der Seite (Sprache, Design) gelten nur für diese Sitzung.
- Colab untersagt auf **kostenlosen** Laufzeiten, das Notebook hauptsächlich über eine
  Web-Oberfläche zu bedienen; solche Laufzeiten können ohne Vorwarnung beendet werden. Die Seite
  ist deshalb als Ansicht neben den Zellen gedacht.

In [ ]:
server = notebook.start(atlas)

## 9. (optional) Den Spiegel in Drive sichern

Eine Archivdatei pro Atlas-Version; Schritt 2 holt sie beim nächsten Mal zurück. Drive lädt im
Hintergrund hoch – erst nach `flush_and_unmount()` ist alles sicher angekommen.

In [ ]:
if USE_DRIVE:
    from google.colab import drive

    print(notebook.backup(DRIVE_DIR))
    drive.flush_and_unmount()

## Chatbots

Der MCP-Endpunkt `/mcp` ist über Colabs Proxy nur für dich im Browser erreichbar, nicht für einen
Chatbot im Netz. Für Claude Desktop oder Claude Code das Paket lokal installieren und `neuron-atlas mcp`
eintragen (README, „Chatbots anbinden“).